# Smoke test of the HBQ quantizer.

In [ ]:
import hierarchical_binary_quantization.hbq as hbq
import torch

## Verify that with 2 rounds of quantization, 

* There are 4 possible values (2 bits) for each element of the latent embedding vectors
* They can be represented by a 2-bit number (0 through 3) that fits comfortably in an int8

In [ ]:
batches, vector_lenght = 3,8
z = torch.randn(batches,vector_lenght)
print(z)
hbq.hbq(z,n_rounds=2)

## Verify that gradient exist and flow through the HBQQuantizer(torch.nn.Module)

In [ ]:
img = [[[-0.7,-0.2],[0.2,0.7]]]
x = torch.tensor([img], requires_grad=True)
quantizer = hbq.HBQQuantizer(n_rounds=2)
q_out, aux = quantizer(x)
loss = (q_out ** 2).sum()
# 4. Compute gradients
loss.backward()
# 5. Print results to verify
print("=== Validation Results ===")
print(f"Original Input x:         {x}")
print(f"Quantized Output q_out:  {q_out}")
print(f"bit_codes:  {aux.bit_codes}")
print(f"Token IDs q_out:  {aux.tokens}")
print(f"Do gradients exist?      {x.grad is not None}")
print(f"Computed Gradients x.grad: {x.grad}")

## Demonstrate that the token component collapses multiple channels into a signle token.

In this case, 5 batches, 3 channels, H=8, W=8, gets quantized into 2 bits per channel, and the 

In [ ]:
z = torch.randn(2,3,4,4)
quantizer = hbq.HBQQuantizer(n_rounds=2)
quantized_latents, q_aux = quantizer(z)
q_aux.tokens.shape
q_aux.tokens

In [ ]:
hbq.tokens_to_bit_codes(torch.tensor([228]),latent_dim=x.shape[-1],n_rounds=2)

In [ ]:
hbq.bit_codes_to_quantized_latent(q_aux.bit_codes,n_rounds=2)